# DataFrames 4: Choosing an engine

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dataframes/dataframes-04-choosing-an-engine.ipynb)

Compare local engines on one report before deciding whether the workload needs distributed execution.

*Last reviewed: November 2026, against pandas 3, Polars 2.1, and DuckDB 2.0.*

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q pandas~=3.0.0 polars~=2.1.0 duckdb~=2.0.0 pyarrow~=26.0

## Setup and sample data

Write one Parquet file with 50,000 rows. Every engine reads that file and returns the same paid revenue by region.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import duckdb
import pandas as pd
import polars as pl

workspace = TemporaryDirectory()
path = Path(workspace.name) / "orders.parquet"
row_count = 50_000
pd.DataFrame({
    "region": ["east", "west"] * (row_count // 2),
    "status": ["paid", "paid", "draft", "draft"] * (row_count // 4),
    "cents": [100, 200, 300, 400] * (row_count // 4),
}).to_parquet(path, index=False)
db = duckdb.connect()
print("input rows:", row_count)

## Run the same report in three engines

Compare complete read, filter and aggregate calls against the same input. Run each several times and show the median.

**Where it matters**

Choose between a pandas workflow, a Polars query and SQL on Parquet.

**Watch out for**

A tiny local run is affected by caching and startup. These timings are observations for this fixture.

In [ ]:
from statistics import median


def pandas_report():
    frame = pd.read_parquet(path, columns=["region", "status", "cents"])
    return list(frame.loc[frame.status.eq("paid")].groupby("region")["cents"].sum().items())


def polars_report():
    return (pl.scan_parquet(path).filter(pl.col("status") == "paid")
            .group_by("region").agg(pl.col("cents").sum()).sort("region").collect().rows())


def duckdb_report():
    return db.execute("""
        select region, sum(cents) from read_parquet(?)
        where status = 'paid' group by region order by region
    """, [str(path)]).fetchall()


for name, report in [("pandas", pandas_report), ("Polars", polars_report), ("DuckDB", duckdb_report)]:
    samples = []
    for _ in range(3):
        start = perf_counter()
        result = report()
        samples.append((perf_counter() - start) * 1000)
    print(name, result, f"median {median(samples):.2f} ms")

**What to notice:** Every engine returns 1,250,000 cents for east and 2,500,000 cents for west.

## Compare the size of materialized data

Measure a materialized input frame and the aggregated result. Use these figures to reason about data shape before sizing a real deployment.

**Where it matters**

Return a small aggregate to Python after filtering and grouping in the engine.

**Watch out for**

Frame size excludes peak process memory, scan buffers and temporary state. Measure peak memory separately for a production benchmark.

In [ ]:
full = pd.read_parquet(path)
summary = db.execute("""
    select region, sum(cents)::bigint as cents from read_parquet(?)
    where status = 'paid' group by region
""", [str(path)]).fetchdf()
print("input:", len(full), "rows,", full.memory_usage(deep=True).sum(), "bytes")
print("output:", len(summary), "rows,", summary.memory_usage(deep=True).sum(), "bytes")

**What to notice:** The Python aggregate has two rows compared with 50,000 input rows. Its measured frame size is much smaller.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

If the result fits in memory but the input does not, name one first attempt before moving to a cluster.

In [ ]:
# Try your solution here.

In [ ]:
print("Scan only needed columns and filters, aggregate inside the engine, and return the small result.")

## Key takeaways

- Choose between a pandas workflow, a Polars query and SQL on Parquet.
- Return a small aggregate to Python after filtering and grouping in the engine.

**Next:** Continue the learning path with [PostgreSQL for data pipelines](../postgres/postgres.ipynb), where pipelines read their source data.